# 02 — Nettoyage Sunbird (reproduit "post-processing" du papier)

Reproduit la pipeline de nettoyage de :
> *Noise mapping and ambient sound recordings of the urban environment in Uganda*
> Nsumba et al., Scientific Data, 2026

Steps (the paper's "post-processing and privacy" section):
1. Chargement du dataset
2. Duplicate removal (metadata hash: submitter + timestamp + GPS)
3. GPS quality filter (accuracy)
4. Statistiques descriptives
5. Saves `sunbird_clean.csv`, used by notebooks 03, 04, 05

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium

# HF token loaded from .env (gitignored)
from dotenv import load_dotenv
import os
load_dotenv('../.env')
HF_TOKEN = os.environ['HF_TOKEN']

ds = load_dataset('Sunbird/urban-noise-uganda-61k', 'small', token=HF_TOKEN)
df = ds['train'].to_pandas()
print(f'Brut : {len(df)} lignes')

## Step 1 - Cleaning (reproduces Section 3 of the paper)

In [ ]:
# Supprime les valeurs manquantes
df = df.dropna(subset=['noise_measurement', 'latitude', 'longitude', 'timestamp'])
print(f'After dropna: {len(df)}')

# Remove duplicates (same collector, same timestamp)
# Limitation identified by the authors: some collectors submitted twice
df = df.drop_duplicates(subset=['submitter_id', 'timestamp'])
print(f'After deduplication: {len(df)}')

# GPS quality filter - keep only accuracy < 50 m
if 'accuracy' in df.columns:
    before = len(df)
    df = df[df['accuracy'] < 50]
    print(f'After GPS accuracy < 50 m filter: {len(df)} ({before - len(df)} removed)')

# Filtre valeurs physiquement impossibles
df = df[(df['noise_measurement'] >= 20) & (df['noise_measurement'] <= 120)]
print(f'After dB [20-120] filter: {len(df)}')

## Step 2 - Descriptive statistics

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()

print('=== Stats globales ===')
print(df['noise_measurement'].describe().round(2))

print('\n=== By region ===')
print(df.groupby('region')['noise_measurement'].describe().round(2))

print('\n=== By noise class ===')
print(df.groupby('class')['noise_measurement'].mean().sort_values(ascending=False).round(2))

## Step 3 - Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution globale
sns.histplot(df['noise_measurement'], bins=40, ax=axes[0,0], color='steelblue')
axes[0,0].axvline(df['noise_measurement'].mean(), color='red', linestyle='--', label=f'Mean={df["noise_measurement"].mean():.1f} dB')
axes[0,0].set_title('Distribution noise_measurement')
axes[0,0].set_xlabel('dB')
axes[0,0].legend()

# By region
sns.boxplot(data=df, x='region', y='noise_measurement', ax=axes[0,1])
axes[0,1].set_title('Noise level by region')

# By hour of day
hourly = df.groupby('hour')['noise_measurement'].mean()
axes[1,0].plot(hourly.index, hourly.values, marker='o', color='steelblue')
axes[1,0].set_title('Mean level by hour')
axes[1,0].set_xlabel('Hour')
axes[1,0].set_ylabel('dB moyen')
axes[1,0].grid(True, alpha=0.3)

# Par classe
class_means = df.groupby('class')['noise_measurement'].mean().sort_values()
class_means.plot(kind='barh', ax=axes[1,1], color='steelblue')
axes[1,1].set_title('Niveau moyen par classe')
axes[1,1].set_xlabel('dB moyen')

plt.suptitle('Reproduction Nsumba et al. 2026 — Urban Noise Uganda', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/sunbird/reproduce_sunbird.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 - Geographic coverage map

In [ ]:
center = [df['latitude'].mean(), df['longitude'].mean()]
m = folium.Map(location=center, zoom_start=12, tiles='CartoDB positron')

def couleur(dB):
    if dB < 55:  return 'green'
    if dB < 70:  return 'orange'
    return 'red'

for _, row in df.sample(min(1000, len(df))).iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=couleur(row['noise_measurement']),
        fill=True,
        fill_opacity=0.7,
        popup=f"{row['noise_measurement']:.1f} dB — {row['class']}"
    ).add_to(m)

m.save('../results/figures/sunbird/sunbird_coverage.html')
print('Map saved -> results/figures/sunbird/sunbird_coverage.html')
m

In [ ]:
# Save the cleaned dataset - input of notebooks 03 (audio QC), 04 (morphology), 05 (figures)
cols = [c for c in ['noise_measurement', 'latitude', 'longitude', 'altitude', 'accuracy',
                    'class', 'class_id', 'region', 'timestamp', 'submitter_id',
                    'hour', 'day_of_week'] if c in df.columns]
df[cols].to_csv('../data/processed/uganda/sunbird_clean.csv', index=False)
print(f'{len(df)} rows saved -> data/processed/uganda/sunbird_clean.csv')

## Summary - what was reproduced

| Paper step | Reproduced |
|---|---|
| Suppression doublons | Oui |
| GPS quality filter | Yes |
| Distribution noise_measurement | Oui |
| Analysis by region | Yes |
| Analyse temporelle (par heure) | Oui |
| Analyse par classe | Oui |
| Carte de couverture | Oui |

**Next step:** apply the same pipeline to our Hanoi field measurements.